# CIPHER — Kaggle Benchmark Notebook
**Calibrated Introspection via Partially Hidden Environment Rules**

Runs the CIPHER metacognition benchmark against as many frontier LLMs as possible
and registers the results on Kaggle Benchmarks.

### Kaggle Dataset required
Upload a dataset named **`cipher-benchmark`** containing:
```
cipher/               ← the Python package
scripts/evaluate.py
data/instances.jsonl  ← 1 000 pre-generated instances (seed=2026)
README.md
```
Then attach it to this notebook at `/kaggle/input/cipher-benchmark/`.

### Scoring dimensions (all normalized to [0, 1])
| Dimension | Weight | What it measures |
|-----------|--------|------------------|
| Objective | 35 % | Final plan quality vs oracle beam search |
| Calibration | 25 % | Brier score on stated self-knowledge |
| Attention | 20 % | Rank correlation on unknown importance |
| Executive | 20 % | Plan structure, named risks, alternative plans |

### API keys (Kaggle Secrets)
Add under **Add-ons → Secrets** before running:
- `MODEL_PROXY_API_KEY` — Kaggle-provisioned $50/day quota, covers all Gemini tiers
- `ANTHROPIC_API_KEY` — optional, enables Claude models
- `OPENAI_API_KEY` — optional, enables GPT-4o family
- `HF_TOKEN` — optional, enables open-source models via HF Inference

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
import subprocess, sys

def _pip(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages], check=True)

_pip("kaggle-benchmarks")
_pip("google-genai")      # Google GenAI SDK (used by kbench)
_pip("anthropic")         # Claude
_pip("openai")            # GPT-4o family
_pip("huggingface_hub")   # open-source models via HF Inference Providers
_pip("tqdm", "pandas", "matplotlib")
print("Dependencies installed.")

In [ ]:
# ── 2. Imports & path setup ───────────────────────────────────────────────────
import os, sys, json, random, time, datetime
from typing import Any, Dict, List, Optional

_KAGGLE_ROOT = "/kaggle/input/datasets/wenhaolu49/cipher"
_LOCAL_ROOT  = os.path.abspath(os.getcwd())       # __file__ doesn't exist in notebooks
_LOCAL_PARENT= os.path.abspath(os.path.join(os.getcwd(), ".."))

CIPHER_ROOT = None
for _c in [_KAGGLE_ROOT, _LOCAL_ROOT, _LOCAL_PARENT]:
    if os.path.isdir(os.path.join(_c, "cipher")):
        CIPHER_ROOT = _c
        break
if CIPHER_ROOT is None:
    raise RuntimeError("cipher/ package not found — attach the 'cipher' dataset.")
if CIPHER_ROOT not in sys.path:
    sys.path.insert(0, CIPHER_ROOT)
print(f"cipher root: {CIPHER_ROOT}")

from cipher import build_prompt
from cipher.generator import Instance
from cipher.world import World, State, EntityState, Rule, Trigger, Effect, Action
from cipher.schema import validate_response
from cipher.scorer import score_response
from cipher.optimal import oracle_score
print("cipher imports OK.")

In [ ]:
# ── 3. Configuration ──────────────────────────────────────────────────────────

DATA_PATH = next(
    (p for p in [
        os.path.join(_KAGGLE_ROOT, "data", "instances.jsonl"),
        os.path.join(CIPHER_ROOT,  "data", "instances.jsonl"),
    ] if os.path.exists(p)),
    None,
)
if DATA_PATH is None:
    raise FileNotFoundError(
        "data/instances.jsonl not found. Generate it first:\n"
        "  python scripts/generate_dataset.py --n 1000 --out data/instances.jsonl --seed 2026 --oracle"
    )

# Number of instances to evaluate per model.
# 50 → fast/cheap sweep.  Set to None for the full 1 000 (final submission).
EVAL_LIMIT   = 50
KBENCH_LIMIT = EVAL_LIMIT   # set to None for the official full-run

def _secret(name: str) -> str:
    if val := os.environ.get(name, ""):
        return val
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return ""

MODEL_PROXY_API_KEY = _secret("MODEL_PROXY_API_KEY")
ANTHROPIC_API_KEY   = _secret("ANTHROPIC_API_KEY")
OPENAI_API_KEY      = _secret("OPENAI_API_KEY")
HF_TOKEN            = _secret("HF_TOKEN")

print(f"DATA_PATH           : {DATA_PATH}")
print(f"EVAL_LIMIT          : {EVAL_LIMIT}")
print(f"MODEL_PROXY_API_KEY : {'✓' if MODEL_PROXY_API_KEY else '✗ MISSING — Gemini/kbench will be skipped'}")
print(f"ANTHROPIC_API_KEY   : {'✓' if ANTHROPIC_API_KEY   else '✗ missing (Claude skipped)'}")
print(f"OPENAI_API_KEY      : {'✓' if OPENAI_API_KEY      else '✗ missing (OpenAI skipped)'}")
print(f"HF_TOKEN            : {'✓' if HF_TOKEN            else '✗ missing (HF models skipped)'}")

In [ ]:
# ── 4. Load instances ─────────────────────────────────────────────────────────

def _load_jsonl(path: str, limit: Optional[int] = None) -> List[Dict]:
    with open(path) as f:
        recs = [json.loads(line) for line in f if line.strip()]
    return recs[:limit] if limit else recs


def _instance_from_record(rec: Dict[str, Any]) -> Instance:
    """Re-hydrate a JSONL record into a live Instance (mirrors scripts/evaluate.py)."""
    h   = rec["hidden"]
    hrl = {e["rule_name"]: set(e["hidden"]) for e in h.get("hidden_fields", [])}
    rules = []
    for r in h["rules"]:
        hides = hrl.get(r["name"], set())
        t, e  = r["trigger"], r["effect"]
        rules.append(Rule(
            name=r["name"],
            trigger=Trigger(
                kind=t["kind"], i=t["i"], j=t.get("j", -1), k=t.get("k", 0),
                hidden_kind=("trigger_kind" in hides),
                hidden_k=("trigger_k" in hides),
            ),
            effect=Effect(
                kind=e["kind"], target=e["target"],
                delta=e.get("delta", 0), source=e.get("source", -1),
                hidden_kind=("effect_kind" in hides),
                hidden_delta=("effect_delta" in hides),
            ),
        ))
    initial = State(tuple(
        EntityState(phase=s["phase"], flux=s["flux"]) for s in h["initial_state"]
    ))
    return Instance(
        id=rec["id"], seed=rec["seed"], difficulty=rec["difficulty"],
        world=World(initial=initial, rules=tuple(rules), horizon=h["horizon"]),
        public_rule_descriptions=[],
        hidden_fields=h.get("hidden_fields", []),
        metacog_ground_truth=h["metacog_ground_truth"],
        true_unknown_ranking=h["true_unknown_ranking"],
        oracle_objective=h.get("oracle_best"),
    )


ALL_RECORDS    = _load_jsonl(DATA_PATH)
EVAL_RECORDS   = _load_jsonl(DATA_PATH, limit=EVAL_LIMIT)
KBENCH_RECORDS = _load_jsonl(DATA_PATH, limit=KBENCH_LIMIT)
print(f"Total: {len(ALL_RECORDS)}  |  eval: {len(EVAL_RECORDS)}  |  kbench: {len(KBENCH_RECORDS)}")

In [ ]:
# ── 5. Scoring helper ─────────────────────────────────────────────────────────

def _extract_json(text: str) -> Dict[str, Any]:
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        return {}
    try:
        return json.loads(text[start : end + 1])
    except json.JSONDecodeError:
        return {}


def score_text_response(raw_text: str, rec: Dict[str, Any]) -> Dict[str, Any]:
    """Parse a model reply string and return the full score-breakdown dict."""
    _zero = dict(composite=0.0, objective=0.0, calibration=0.0, attention=0.0, executive=0.0)
    raw = _extract_json(raw_text)
    if not raw:
        return {**_zero, "error": "no_json"}
    inst = _instance_from_record(rec)
    resp = validate_response(raw)
    bd   = score_response(resp, inst,
                          best_obj=rec["hidden"].get("oracle_best"),
                          worst_obj=rec["hidden"].get("oracle_worst"))
    return bd.to_dict()

In [ ]:
# ── 6. Stub agents (no API key needed) ───────────────────────────────────────

def stub_noop(inst: Instance) -> Dict[str, Any]:
    return {
        "metacog_assessment": [
            {"rule_name": g["rule_name"], "component": g["component"],
             "known": True, "confidence": 0.5}
            for g in inst.metacog_ground_truth
        ],
        "critical_unknowns_ranked": [],
        "exploratory_actions": [],
        "final_plan": [{"kind": "wait"}],
        "self_judgment": {"robustness_score": 50, "risks_identified": [],
                          "alternative_if_unknown_X": {}},
    }


def stub_random(inst: Instance) -> Dict[str, Any]:
    rng = random.Random(inst.seed ^ 0xDEADBEEF)
    n   = inst.world.initial.n
    kds = ["pulse", "damp", "shift", "unshift", "observe", "wait"]
    act = lambda: {"kind": rng.choice(kds), "i": rng.randrange(n)}
    return {
        "metacog_assessment": [
            {"rule_name": g["rule_name"], "component": g["component"],
             "known": rng.random() > 0.5, "confidence": rng.random()}
            for g in inst.metacog_ground_truth
        ],
        "critical_unknowns_ranked": list(inst.true_unknown_ranking[::-1]),
        "exploratory_actions": [act() for _ in range(rng.randint(0, 2))],
        "final_plan":          [act() for _ in range(rng.randint(1, 5))],
        "self_judgment": {"robustness_score": 40,
                          "risks_identified": ["random baseline"],
                          "alternative_if_unknown_X": {"unknown": "", "plan": [act()]}},
    }


def stub_greedy(inst: Instance) -> Dict[str, Any]:
    """Beam-search the visible world — strong objective, weak calibration."""
    _, best_plan = oracle_score(inst.world)
    return {
        "metacog_assessment": [
            {"rule_name": g["rule_name"], "component": g["component"],
             "known": True, "confidence": 0.9}
            for g in inst.metacog_ground_truth
        ],
        "critical_unknowns_ranked": [],
        "exploratory_actions": [],
        "final_plan": [{"kind": a.kind, "i": a.i, "j": a.j} for a in best_plan],
        "self_judgment": {"robustness_score": 70, "risks_identified": [],
                          "alternative_if_unknown_X": {}},
    }


print("Stub agents ready: noop / random / greedy")

In [ ]:
# ── 7. Gemini agents (Google GenAI SDK) ──────────────────────────────────────

def _make_gemini_agent(model_id: str):
    def _agent(inst: Instance) -> Dict[str, Any]:
        import google.genai as genai
        from google.genai import types as gtypes
        resp = genai.Client(api_key=MODEL_PROXY_API_KEY).models.generate_content(
            model=model_id,
            contents=build_prompt(inst),
            config=gtypes.GenerateContentConfig(max_output_tokens=2048, temperature=0.0),
        )
        return _extract_json(resp.text or "")
    _agent.__name__ = model_id
    return _agent


GEMINI_MODELS = [
    "gemini-2.5-pro",
    "gemini-2.0-flash",
    "gemini-2.0-flash-lite",
    "gemini-1.5-pro",
    "gemini-1.5-flash",
    "gemini-1.5-flash-8b",
]

gemini_agents = {m: _make_gemini_agent(m) for m in GEMINI_MODELS} if MODEL_PROXY_API_KEY else {}
print(f"Gemini : {list(gemini_agents) or ['(skipped — no MODEL_PROXY_API_KEY)']}")

In [ ]:
# ── 8. Claude agents (Anthropic API) ─────────────────────────────────────────

def _make_claude_agent(model_id: str):
    def _agent(inst: Instance) -> Dict[str, Any]:
        import anthropic
        msg = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY).messages.create(
            model=model_id, max_tokens=2048,
            messages=[{"role": "user", "content": build_prompt(inst)}],
        )
        text = "".join(b.text for b in msg.content if getattr(b, "type", "") == "text")
        return _extract_json(text)
    _agent.__name__ = model_id
    return _agent


CLAUDE_MODELS = [
    "claude-opus-4-6",
    "claude-sonnet-4-6",
    "claude-haiku-4-5-20251001",
]

claude_agents = {m: _make_claude_agent(m) for m in CLAUDE_MODELS} if ANTHROPIC_API_KEY else {}
print(f"Claude : {list(claude_agents) or ['(skipped — no ANTHROPIC_API_KEY)']}")

In [ ]:
# ── 9. OpenAI agents ─────────────────────────────────────────────────────────

def _make_openai_agent(model_id: str):
    _reasoning = model_id.startswith(("o1", "o3"))  # o-series don't take temperature
    def _agent(inst: Instance) -> Dict[str, Any]:
        from openai import OpenAI
        kwargs: Dict[str, Any] = {
            "model": model_id,
            "max_completion_tokens": 2048,
            "messages": [{"role": "user", "content": build_prompt(inst)}],
        }
        if not _reasoning:
            kwargs["temperature"] = 0.0
        resp = OpenAI(api_key=OPENAI_API_KEY).chat.completions.create(**kwargs)
        return _extract_json(resp.choices[0].message.content or "")
    _agent.__name__ = model_id
    return _agent


OPENAI_MODELS = ["gpt-4o", "gpt-4o-mini", "o1-mini", "o3-mini"]

openai_agents = {m: _make_openai_agent(m) for m in OPENAI_MODELS} if OPENAI_API_KEY else {}
print(f"OpenAI : {list(openai_agents) or ['(skipped — no OPENAI_API_KEY)']}")

In [ ]:
# ── 10. HuggingFace Inference Provider agents ─────────────────────────────────

def _make_hf_agent(model_id: str):
    _short = model_id.split("/")[-1]
    def _agent(inst: Instance) -> Dict[str, Any]:
        from huggingface_hub import InferenceClient
        resp = InferenceClient(model=model_id, token=HF_TOKEN).chat_completion(
            messages=[{"role": "user", "content": build_prompt(inst)}],
            max_tokens=2048, temperature=0.0,
        )
        return _extract_json(resp.choices[0].message.content or "")
    _agent.__name__ = _short
    return _agent


HF_MODELS = [
    "meta-llama/Llama-3.3-70B-Instruct",
    "meta-llama/Llama-3.1-8B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "mistralai/Mixtral-8x7B-Instruct-v0.1",
    "Qwen/Qwen2.5-72B-Instruct",
    "deepseek-ai/DeepSeek-R1-Distill-Llama-70B",
]

hf_agents = {m.split("/")[-1]: _make_hf_agent(m) for m in HF_MODELS} if HF_TOKEN else {}
print(f"HF     : {list(hf_agents) or ['(skipped — no HF_TOKEN)']}")

In [ ]:
# ── 11. Master registry ───────────────────────────────────────────────────────

STUB_AGENTS = {"stub-noop": stub_noop, "stub-random": stub_random, "stub-greedy": stub_greedy}

ALL_AGENTS: Dict[str, Any] = {
    **STUB_AGENTS,
    **gemini_agents,
    **claude_agents,
    **openai_agents,
    **hf_agents,
}

print(f"{len(ALL_AGENTS)} agents registered:")
for k in ALL_AGENTS:
    print(f"  · {k}")

In [ ]:
# ── 12. Local evaluation loop (all models) ────────────────────────────────────
# Runs every registered agent locally and builds the leaderboard table.
# Results are shown in notebook output but are NOT registered on Kaggle Benchmarks
# (that happens in cells 17-20 via cipher_task.run()).

from tqdm.auto import tqdm

def evaluate_agent(name: str, agent_fn, records: List[Dict]) -> Dict[str, Any]:
    per_instance, n_errors = [], 0
    for rec in tqdm(records, desc=f"{name[:40]:40s}", leave=False):
        inst = _instance_from_record(rec)
        try:
            raw  = agent_fn(inst)
            resp = validate_response(raw)
            bd   = score_response(resp, inst,
                                  best_obj=rec["hidden"].get("oracle_best"),
                                  worst_obj=rec["hidden"].get("oracle_worst"))
            per_instance.append({"id": inst.id, "difficulty": inst.difficulty, **bd.to_dict()})
        except Exception as exc:
            n_errors += 1
            per_instance.append({"id": inst.id, "difficulty": inst.difficulty,
                                  "composite": 0.0, "objective": 0.0, "calibration": 0.0,
                                  "attention": 0.0, "executive": 0.0, "error": str(exc)})
    n   = len(per_instance)
    avg = lambda k: sum(r.get(k, 0.0) for r in per_instance) / max(n, 1)
    return {
        "agent": name, "n": n, "n_errors": n_errors,
        "mean_composite":   avg("composite"),
        "mean_objective":   avg("objective"),
        "mean_calibration": avg("calibration"),
        "mean_attention":   avg("attention"),
        "mean_executive":   avg("executive"),
        "per_instance":     per_instance,
        "timestamp":        datetime.datetime.utcnow().isoformat(timespec="seconds") + "Z",
    }


RESULTS: Dict[str, Dict] = {}
for _name, _fn in ALL_AGENTS.items():
    print(f"\nEvaluating {_name} …")
    _t0 = time.time()
    _r  = evaluate_agent(_name, _fn, EVAL_RECORDS)
    RESULTS[_name] = _r
    print(f"  composite={_r['mean_composite']:.3f}  obj={_r['mean_objective']:.3f}  "
          f"cal={_r['mean_calibration']:.3f}  att={_r['mean_attention']:.3f}  "
          f"exe={_r['mean_executive']:.3f}  errors={_r['n_errors']}  ({time.time()-_t0:.1f}s)")

print("\n✓ All local evaluations complete.")

In [ ]:
# ── 13. Leaderboard table ─────────────────────────────────────────────────────

import pandas as pd

_lb_rows = [
    {
        "Model":       r["agent"],
        "Composite ↑": round(r["mean_composite"],   3),
        "Objective":   round(r["mean_objective"],    3),
        "Calibration": round(r["mean_calibration"],  3),
        "Attention":   round(r["mean_attention"],    3),
        "Executive":   round(r["mean_executive"],    3),
        "N":           r["n"],
        "Errors":      r["n_errors"],
    }
    for r in sorted(RESULTS.values(), key=lambda x: -x["mean_composite"])
]

df_lb = pd.DataFrame(_lb_rows)
display(df_lb)

In [ ]:
# ── 14. Visualisation ─────────────────────────────────────────────────────────

import matplotlib.pyplot as plt
import numpy as np

models     = df_lb["Model"].tolist()
composites = df_lb["Composite ↑"].tolist()
DIMS       = ["Objective", "Calibration", "Attention", "Executive"]

fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(17, max(5, len(models) * 0.5)))

# Left: composite bar chart
y    = np.arange(len(models))
bars = ax_l.barh(y, composites, color="steelblue", edgecolor="white", height=0.65)
ax_l.set_yticks(y); ax_l.set_yticklabels(models, fontsize=9)
ax_l.set_xlabel("Composite score (0 – 1)")
ax_l.set_title("CIPHER Leaderboard — Composite", fontweight="bold")
ax_l.set_xlim(0, 1.05)
ax_l.axvline(0.5, color="red", lw=1, ls="--", alpha=0.5, label="pass threshold")
ax_l.legend(fontsize=8)
for bar, val in zip(bars, composites):
    ax_l.text(bar.get_width() + 0.012, bar.get_y() + bar.get_height()/2,
              f"{val:.3f}", va="center", fontsize=8)
ax_l.invert_yaxis()

# Right: sub-score breakdown
x     = np.arange(len(DIMS))
width = min(0.8 / max(len(models), 1), 0.12)
cmap  = plt.get_cmap("tab20")
for idx, model in enumerate(models):
    row    = df_lb[df_lb["Model"] == model].iloc[0]
    scores = [float(row[d]) for d in DIMS]
    offset = (idx - len(models)/2 + 0.5) * width
    ax_r.bar(x + offset, scores, width, label=model,
             color=cmap(idx / max(len(models), 1)), alpha=0.85)
ax_r.set_xticks(x); ax_r.set_xticklabels(DIMS)
ax_r.set_ylim(0, 1.0); ax_r.set_ylabel("Score")
ax_r.set_title("Sub-score Breakdown", fontweight="bold")
ax_r.legend(fontsize=7, loc="upper right", ncol=max(1, len(models)//8))

plt.tight_layout()
os.makedirs("/kaggle/working", exist_ok=True)
_chart = "/kaggle/working/cipher_leaderboard.png"
plt.savefig(_chart, dpi=150, bbox_inches="tight")
plt.show()
print(f"Chart saved → {_chart}")

In [ ]:
# ── 15. Breakdown by difficulty ───────────────────────────────────────────────

diff_rows = []
for name, result in RESULTS.items():
    by_diff: Dict[str, List[float]] = {"easy": [], "medium": [], "hard": []}
    for r in result["per_instance"]:
        by_diff.get(r.get("difficulty", "medium"), []).append(r.get("composite", 0.0))
    diff_rows.append({
        "Model":  name,
        "Easy":   round(sum(by_diff["easy"])   / max(len(by_diff["easy"]),   1), 3),
        "Medium": round(sum(by_diff["medium"]) / max(len(by_diff["medium"]), 1), 3),
        "Hard":   round(sum(by_diff["hard"])   / max(len(by_diff["hard"]),   1), 3),
    })
df_diff = pd.DataFrame(sorted(diff_rows, key=lambda r: -(r["Easy"]+r["Medium"]+r["Hard"])))
print("Composite by difficulty:")
display(df_diff)

In [ ]:
# ── 16. Save full results JSON ────────────────────────────────────────────────

os.makedirs("/kaggle/working", exist_ok=True)
_out = "/kaggle/working/cipher_all_results.json"
with open(_out, "w") as _f:
    json.dump({
        "leaderboard":   _lb_rows,
        "by_difficulty": diff_rows,
        "per_model": {
            k: {"summary":      {kk: vv for kk, vv in v.items() if kk != "per_instance"},
                "per_instance": v["per_instance"]}
            for k, v in RESULTS.items()
        },
    }, _f, indent=2)
print(f"Full results → {_out}")

---
## Kaggle Benchmarks — Official Task Registration

The cells below use the `kaggle-benchmarks` SDK to register every model run
on the **Kaggle Benchmarks platform** (not just notebook output).

**How it works:**
- One `@kbench.task` is defined: `cipher_metacog`
- `cipher_task.run(llm=..., ...)` is called once per instance per model
- A run **passes** when `composite ≥ 0.5`; the platform shows the pass-rate
- All models — Gemini, Claude, OpenAI, HF, and stubs — are wired in

**After running:** go to https://www.kaggle.com/benchmarks/tasks/new,
publish `cipher_metacog`, create a Benchmark collection, and attach the
URL to your writeup.

In [ ]:
# ── 17. kbench task definition ────────────────────────────────────────────────

import kaggle_benchmarks as kbench
from kaggle_benchmarks.actors.llms import GoogleGenAI
import google.genai as genai
from google.genai import types as gtypes


@kbench.task(name="cipher_metacog")
def cipher_task(llm, instance_id: str, prompt: str, record_json: str):
    """
    CIPHER — Calibrated Introspection via Partially Hidden Environment Rules.

    The LLM receives a partial causal world in invented vocabulary and must
    return strict JSON with: metacog_assessment · critical_unknowns_ranked ·
    exploratory_actions · final_plan · self_judgment.

    Scored: Objective (35%) · Calibration (25%) · Attention (20%) · Executive (20%).
    Passes when composite ≥ 0.5.
    """
    reply     = llm.prompt(prompt)
    rec       = json.loads(record_json)
    breakdown = score_text_response(str(reply), rec)
    composite = breakdown.get("composite", 0.0)
    kbench.assertions.assert_true(
        composite >= 0.5,
        f"[{instance_id}] composite {composite:.3f} < 0.5 — "
        f"obj={breakdown.get('objective',0):.3f} "
        f"cal={breakdown.get('calibration',0):.3f} "
        f"att={breakdown.get('attention',0):.3f} "
        f"exe={breakdown.get('executive',0):.3f}",
    )


print(f"cipher_task defined — {len(KBENCH_RECORDS)} instances per model.")

In [ ]:
# ── 18a. kbench LLM actor wrappers ───────────────────────────────────────────
# kaggle-benchmarks ships GoogleGenAI out of the box.
# For every other provider we create a thin wrapper that implements
# the same interface: a class with a .prompt(text) -> str method.

class _ClaudeActor:
    """Calls Anthropic Claude via the official SDK."""
    def __init__(self, model_id: str):
        import anthropic
        self._client   = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        self._model_id = model_id

    def prompt(self, text: str) -> str:
        msg = self._client.messages.create(
            model=self._model_id, max_tokens=2048,
            messages=[{"role": "user", "content": text}],
        )
        return "".join(b.text for b in msg.content if getattr(b, "type", "") == "text")


class _OpenAIActor:
    """Calls OpenAI via the official SDK."""
    def __init__(self, model_id: str):
        from openai import OpenAI
        self._client   = OpenAI(api_key=OPENAI_API_KEY)
        self._model_id = model_id
        self._reasoning= model_id.startswith(("o1", "o3"))

    def prompt(self, text: str) -> str:
        kwargs: Dict[str, Any] = {
            "model": self._model_id,
            "max_completion_tokens": 2048,
            "messages": [{"role": "user", "content": text}],
        }
        if not self._reasoning:
            kwargs["temperature"] = 0.0
        resp = self._client.chat.completions.create(**kwargs)
        return resp.choices[0].message.content or ""


class _HFActor:
    """Calls a HuggingFace model via InferenceClient."""
    def __init__(self, model_id: str):
        from huggingface_hub import InferenceClient
        self._client = InferenceClient(model=model_id, token=HF_TOKEN)

    def prompt(self, text: str) -> str:
        resp = self._client.chat_completion(
            messages=[{"role": "user", "content": text}],
            max_tokens=2048, temperature=0.0,
        )
        return resp.choices[0].message.content or ""


class _StubActor:
    """Wraps a stub function as a kbench actor.
    Must call .set_rec(rec) before each .prompt() so the stub has the Instance.
    """
    def __init__(self, stub_fn):
        self._fn  = stub_fn
        self._rec: Optional[Dict] = None

    def set_rec(self, rec: Dict) -> None:
        self._rec = rec

    def prompt(self, _text: str) -> str:
        if self._rec is None:
            return "{}"
        return json.dumps(self._fn(_instance_from_record(self._rec)))


print("Actor wrappers defined: _ClaudeActor / _OpenAIActor / _HFActor / _StubActor")

In [ ]:
# ── 18b. Run kbench — Gemini models ──────────────────────────────────────────

if not MODEL_PROXY_API_KEY:
    print("Skipping Gemini kbench — MODEL_PROXY_API_KEY not set.")
else:
    _gc = genai.Client(api_key=MODEL_PROXY_API_KEY)
    for _mid in GEMINI_MODELS:
        print(f"\n── kbench · {_mid} ({len(KBENCH_RECORDS)} instances) ──")
        _llm = GoogleGenAI(client=_gc, model=_mid)
        for _rec in tqdm(KBENCH_RECORDS, desc=_mid, leave=False):
            try:
                cipher_task.run(llm=_llm, instance_id=_rec["id"],
                                prompt=_rec["prompt"], record_json=json.dumps(_rec))
            except Exception as _e:
                print(f"  ⚠ {_rec['id']}: {_e}")
        print(f"  ✓ done")
    print("\n✓ Gemini kbench runs complete.")

In [ ]:
# ── 19a. Run kbench — Claude models ──────────────────────────────────────────

if not ANTHROPIC_API_KEY:
    print("Skipping Claude kbench — ANTHROPIC_API_KEY not set.")
else:
    for _mid in CLAUDE_MODELS:
        print(f"\n── kbench · {_mid} ({len(KBENCH_RECORDS)} instances) ──")
        _llm = _ClaudeActor(_mid)
        for _rec in tqdm(KBENCH_RECORDS, desc=_mid, leave=False):
            try:
                cipher_task.run(llm=_llm, instance_id=_rec["id"],
                                prompt=_rec["prompt"], record_json=json.dumps(_rec))
            except Exception as _e:
                print(f"  ⚠ {_rec['id']}: {_e}")
        print(f"  ✓ done")
    print("\n✓ Claude kbench runs complete.")

In [ ]:
# ── 19b. Run kbench — OpenAI models ──────────────────────────────────────────

if not OPENAI_API_KEY:
    print("Skipping OpenAI kbench — OPENAI_API_KEY not set.")
else:
    for _mid in OPENAI_MODELS:
        print(f"\n── kbench · {_mid} ({len(KBENCH_RECORDS)} instances) ──")
        _llm = _OpenAIActor(_mid)
        for _rec in tqdm(KBENCH_RECORDS, desc=_mid, leave=False):
            try:
                cipher_task.run(llm=_llm, instance_id=_rec["id"],
                                prompt=_rec["prompt"], record_json=json.dumps(_rec))
            except Exception as _e:
                print(f"  ⚠ {_rec['id']}: {_e}")
        print(f"  ✓ done")
    print("\n✓ OpenAI kbench runs complete.")

In [ ]:
# ── 19c. Run kbench — HuggingFace models ─────────────────────────────────────

if not HF_TOKEN:
    print("Skipping HF kbench — HF_TOKEN not set.")
else:
    for _mid in HF_MODELS:
        _short = _mid.split("/")[-1]
        print(f"\n── kbench · {_short} ({len(KBENCH_RECORDS)} instances) ──")
        _llm = _HFActor(_mid)
        for _rec in tqdm(KBENCH_RECORDS, desc=_short, leave=False):
            try:
                cipher_task.run(llm=_llm, instance_id=_rec["id"],
                                prompt=_rec["prompt"], record_json=json.dumps(_rec))
            except Exception as _e:
                print(f"  ⚠ {_rec['id']}: {_e}")
        print(f"  ✓ done")
    print("\n✓ HF kbench runs complete.")

In [ ]:
# ── 19d. Run kbench — stub baselines ─────────────────────────────────────────
# Stubs need the Instance object; _StubActor.set_rec() injects it before each call.

for _sname, _sfn in STUB_AGENTS.items():
    print(f"\n── kbench · {_sname} ({len(KBENCH_RECORDS)} instances) ──")
    _llm = _StubActor(_sfn)
    for _rec in tqdm(KBENCH_RECORDS, desc=_sname, leave=False):
        _llm.set_rec(_rec)
        try:
            cipher_task.run(llm=_llm, instance_id=_rec["id"],
                            prompt=_rec["prompt"], record_json=json.dumps(_rec))
        except Exception:
            pass  # stubs are expected to fail the composite >= 0.5 threshold
    print(f"  ✓ done")

print("\n✓ All kbench runs complete.")

In [ ]:
# ── 20. Final summary ─────────────────────────────────────────────────────────

print("=" * 72)
print("CIPHER — FINAL LEADERBOARD")
print("=" * 72)
print(df_lb.to_string(index=False))
print()
print("Next steps:")
print("  1. Set KBENCH_LIMIT = None and re-run cells 17–19 for all 1 000 instances.")
print("  2. https://www.kaggle.com/benchmarks/tasks/new → publish 'cipher_metacog'.")
print("  3. Create Benchmark collection 'CIPHER', add the task.")
print("  4. Paste benchmark URL into your writeup under Project Links.")
print("  5. Add a cover image and submit the writeup before the deadline.")